In [35]:
# langgraph는 checkpoint를 가지고 있기에 해당 체크포인트로 돌아갈 수도 있고 그 체크포인트를 수정할 수도 있다
import sqlite3
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.sqlite import SqliteSaver

llm = init_chat_model("openai:gpt-4o-mini")

conn = sqlite3.connect("memory.db", check_same_thread=False)

config = {"configurable": {"thread_id": "3"}}

In [ ]:
# MessagesState 안에는 add_messages 함수가 있는데 여기에 내가 원하는 대화의 ID를 넣어주면 메시지를 추가하거나 수정할 수 있음
class State(MessagesState):
    pass

graph_builder = StateGraph(State)

In [37]:
def chatbot(state: State):
    response = llm.invoke(state["messages"])
    return {
        "messages": [response]
    }

In [38]:
graph_builder.add_node("chatbot", chatbot)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

graph = graph_builder.compile(
    checkpointer=SqliteSaver(conn)
)

In [39]:
result = graph.invoke(
    { "messages": [{"role": "user", "content": "I live in Korea now. And the city I live in is seoul"}] },
    config=config
)

In [40]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Hello my name is lucas
================================== Ai Message ==================================

Hello Lucas! How can I assist you today?
================================ Human Message =================================

I live in Korea now. And the city I live in is seoul
================================== Ai Message ==================================

That's great! Seoul is a vibrant city with a rich culture and a lot to offer. Do you have any favorite places or activities in Seoul? Or is there something specific you’d like to know or discuss about the city?


In [44]:
# 만약 내가 살고 있는 도시를 수정하고 싶다면
# 1. state_history를 가져오는게 필요하다
# 여기서 config를 넣어줘야 한다. config는 thread_id를 가지고 있음 이 thread_id를 가지고 LangGraph가 지금 어떤 대화를 언급하는지 알 수 있음
state_history = graph.get_state_history(config)

In [45]:
for state_snapshot in list(state_history):
    print(state_snapshot.next)
    print(state_snapshot.values["messages"])
    print("==========\n")

()
[HumanMessage(content='Hello my name is lucas', additional_kwargs={}, response_metadata={}, id='fd75c118-6378-4880-a182-7c1bab559e93'), AIMessage(content='Hello Lucas! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 13, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_80d7a26d20', 'id': 'chatcmpl-DLz5ppDC1JT7VcCo5L5lox6VJAqLd', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1283-7d3e-74b1-9596-5fb0f968a24e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 10, 'total_tokens': 23, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output

In [ ]:
state_history = graph.get_state_history(config)
# -5를 붙인 이유는 뒤에서부터 가장 최신의 대화가 있기에 원하는 대화를 뽑기 위함임
to_fork = list(state_history)[-5]

to_fork.values["messages"]

[HumanMessage(content='Hello my name is lucas', additional_kwargs={}, response_metadata={}, id='fd75c118-6378-4880-a182-7c1bab559e93'),
 AIMessage(content='Hello Lucas! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 13, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_80d7a26d20', 'id': 'chatcmpl-DLz5ppDC1JT7VcCo5L5lox6VJAqLd', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1283-7d3e-74b1-9596-5fb0f968a24e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 10, 'total_tokens': 23, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_t

In [47]:
from langchain_core.messages import HumanMessage

graph.update_state(
    # 이렇게하면 내가 원하는 지점의 config를 얻을 수 있음 해당 config안에 내가 원하는 지점의 checkpoint_id가 있음
    to_fork.config,
    {
        "messages": [HumanMessage(
            content="I live in Korea now. And the city I live in is anyang",
            id="44ceb8ee-6cdf-4cce-84c8-e6aaad91ad00"
        )]
    }
)

{'configurable': {'thread_id': '3',
  'checkpoint_ns': '',
  'checkpoint_id': '1f125783-bb5f-643e-8004-03ca3e8dbfa7'}}

In [48]:
forked_state = graph.get_state_history({'configurable': {'thread_id': '3','checkpoint_ns': '','checkpoint_id': '1f125783-bb5f-643e-8004-03ca3e8dbfa7'}})

list(forked_state)

[StateSnapshot(values={'messages': [HumanMessage(content='Hello my name is lucas', additional_kwargs={}, response_metadata={}, id='fd75c118-6378-4880-a182-7c1bab559e93'), AIMessage(content='Hello Lucas! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 13, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_80d7a26d20', 'id': 'chatcmpl-DLz5ppDC1JT7VcCo5L5lox6VJAqLd', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1283-7d3e-74b1-9596-5fb0f968a24e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 10, 'total_tokens': 23, 'input_token_details': {'audi

In [ ]:
# 이렇게 해줌으로 해당 시점에서부터 대화를 다시 시작할 수 있음
result = graph.invoke(None, {'configurable': {'thread_id': '3','checkpoint_ns': '','checkpoint_id': '1f125783-bb5f-643e-8004-03ca3e8dbfa7'}})

for message in result["messages"]:
    message.pretty_print()